# Full protocol text as context (split questions)

**One factor changed relative to [08c_split_questions_protocol_criteria.ipynb](08c_split_questions_protocol_criteria.ipynb):** `state.review.protocol` holds the protocol's full text (Background, Objectives, Methods; median about 27k characters). The four per-section questions are unchanged (protocol criteria sections, same sentence). Before, `state` held only the record.

Source: same pages as the protocol-criteria run ([scripts/fulltext_criteria.py](../../scripts/fulltext_criteria.py), `extract_protocol_text`). For CD000996 and CD001261 (previous review versions, no protocol online) the text also stops before *Results*, so no included studies of the earlier version leak in.

Just for exploration, on the subset only.

Labels: CLEF TAR abstract-level (`label_abs`) and content-level (`label_content`).

Nothing paid runs unless `RUN_SMOKE`, `RUN_SUBSET` or `RUN_PAID` is set to `True` below.

In [1]:
import json, sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
import screening as S

RUN_SMOKE = False    # one paid call
RUN_SUBSET = True    # the 10 reviews with < 1000 records (3,637 records); see the cost estimate
RUN_PAID = False     # all 20 reviews; read the cost estimate first
OUT = S.out_dir("protocol_fulltext_context_split_questions")
FINAL = S.out_dir("protocol_criteria_split_questions")      # same questions, no protocol text in the state
REFERENCE = {"baseline": S.out_dir("baseline"), "title_abstract_no_results": S.out_dir("review_title_abstract_no_results")}

/Users/joshua/Desktop/Repositories/Research/decision-model-screener/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data and split

In [2]:
records, CRITERIA = S.load_records()
records = S.add_bm25(records, CRITERIA)
PROTO = json.loads((ROOT / "data" / "processed" / "tar2019_intervention_protocol_criteria.json").read_text())
FT = PROTO
FULL = json.loads((ROOT / "data" / "processed" / "tar2019_intervention_protocol_fulltext.json").read_text())
assert set(FULL) == set(CRITERIA)
assert set(PROTO) == set(CRITERIA)

SECTIONS = {"studies": "Types of studies", "participants": "Types of participants",
            "interventions": "Types of interventions", "outcomes": "Types of outcome measures"}

def split(criteria):
    """{question key: section text without its heading}, split at the '#### Types of ...' headings."""
    parts, key = {}, None
    for line in criteria.split("\n")[1:]:
        head = line.lstrip("#").strip()
        if line.startswith("#### ") and head in SECTIONS.values():
            key = next(k for k, v in SECTIONS.items() if v == head)
            parts[key] = []
        else:
            parts[key].append(line)
    return {k: "\n".join(v).strip() for k, v in parts.items()}

PARTS = {r: split(v["criteria"]) for r, v in PROTO.items()}
FINAL_PARTS = {r: split(v["criteria"]) for r, v in FT.items()}
assert all(set(p) == set(SECTIONS) and all(p.values()) for p in PARTS.values()), "a protocol lacks a section"

n = records.groupby("review").size()
SUBSET = sorted(n[n < 1000].index)
print("subset:", len(SUBSET), "reviews,", int(n[SUBSET].sum()), "records")
pd.DataFrame({"type": {r: v["type"] for r, v in PROTO.items()}, "protocol_text_chars": {r: len(v["text"]) for r, v in FULL.items()},
              "in_subset": {r: r in SUBSET for r in PROTO}})

20 reviews, 41996 records, 41183 unique PMIDs
not returned by PubMed: 0 | no abstract: 2183 of 41996


subset: 10 reviews, 3637 records


,type,protocol_text_chars,in_subset
CD006468,Protocol,19935,False
CD010753,Protocol,25913,False
CD011571,Protocol,26031,True
CD011977,Protocol,29214,True
CD012342,Protocol,40233,False
CD009642,Protocol,20631,False
CD011768,Protocol,46990,False
CD010038,Protocol,42090,False
CD010239,Protocol,24247,True
CD011140,Protocol,25251,True


## Decision model input

Same questions as the protocol-criteria run; the state adds the protocol text.

In [3]:
def build_body(review, title, abstract):
    return {
        "model": S.MODEL,
        "state": {"review": {"protocol": FULL[review]["text"]},
                  "record": {"title": title, "abstract": abstract}},
        "questions": {
            k: {"type": "noul",
                "instructions": f"The record meets this eligibility criterion of the systematic review ({SECTIONS[k].lower()}): "
                                f"it meets the stated requirements and none of the stated exclusions.\n\n{PARTS[review][k]}"}
            for k in SECTIONS
        },
    }

body = build_body(*records.loc[0, ["review", "title", "abstract"]])
print(json.dumps(body, indent=1)[:1500])
print("...")
print("questions:", json.dumps(body["questions"], indent=1)[:1500])

{
 "model": "typesafe/jev-1.14",
 "state": {
  "review": {
   "protocol": "## Background\nBronchiectasis, previously termed an 'orphan disease' is increasingly recognized as a major cause of respiratory morbidity especially in developing countries (Karadag 2005, Karakoc 2001) and in some ethnic populations of affluent countries (Chang 2002, Edwards 2003, Singleton 2000). The underlying aetiology of bronchiectasis varies from post recurrent respiratory infections to rare immune deficiencies. Other causes include primary ciliary dyskinesia, allergic bronchopulmonary aspergillosis and Mycobacterial infection (Shoemark 2007). However, bronchiectasis is also a common pathway for a variety of diseases. Thus, the presence of bronchiectasis is also increasingly recognised in common (e.g. chronic obstructive pulmonary disease (COPD) (Patel 2004, O'Brien 2000) and uncommon respiratory diseases (e.g. bronchiolitis obliterans (Chang 1998) and sarcoidosis (Lewis 2002)) as well non primary respirato

## Cost estimate (no API call)

In [4]:
est = S.estimate_cost(records, build_body)
print(f"subset ({len(SUBSET)} reviews): ${est.loc[SUBSET, 'usd'].sum():.2f}")

TypeSafe | typesafe/jev-1.14-20261014 | USD per 1M input tokens: 0.041999999999999996 | output: 0


requests: 41996 | mean chars per request: 42247 | max: 62010
estimated input tokens: 579.8M | estimated cost: $24.35
subset (10 reviews): $1.39


## Smoke test (one paid call)

In [5]:
if RUN_SMOKE:
    big = max(SUBSET, key=lambda r: len(FULL[r]["text"]))      # longest protocol in the subset: checks the API accepts it
    rec = records[records.review == big].iloc[0]
    print(big, len(FULL[big]["text"]), "chars of protocol text")
    print(json.dumps(S.call(build_body(rec.review, rec.title, rec.abstract)), indent=1))

## Subset (paid)

In [6]:
if RUN_SUBSET:
    S.run_reviews(records, SUBSET, build_body, OUT)

reviews:   0%|          | 0/10 [00:00<?, ?it/s]

CD000996:   0%|          | 0/281 [00:00<?, ?it/s]

CD000996:   0%|          | 1/281 [00:00<01:14,  3.78it/s]

CD000996:   2%|▏         | 6/281 [00:00<00:14, 19.62it/s]

CD000996:   3%|▎         | 9/281 [00:00<00:19, 14.24it/s]

CD000996:   4%|▍         | 12/281 [00:00<00:22, 11.94it/s]

... (1,478 outputs left out)

CD012551:  99%|█████████▉| 588/591 [01:00<00:00, 12.11it/s]

CD012551: 100%|█████████▉| 590/591 [01:00<00:00, 10.90it/s]

reviews: 100%|██████████| 10/10 [05:17<00:00, 32.62s/it]

reviews: 100%|██████████| 10/10 [05:17<00:00, 31.74s/it]

## Full run (paid)

In [7]:
if RUN_PAID:
    S.run_reviews(records, CRITERIA, build_body, OUT)

## Results

Protocol vs final criteria on the same records: the reviews with a cache file for this run (the final-criteria split run covers all 20). LGAR baseline and title + abstract without results for reference. A failed call or a missing answer gives the lowest score (counted below).

In [8]:
done = sorted(p.stem for p in OUT.glob("*.jsonl"))
print(len(done), "of", len(CRITERIA), "reviews run")
sub = records[records.review.isin(done)].copy()
for col, path in REFERENCE.items():
    sub = S.add_scores(sub, path, col)

def load_split(path, prefix):
    rows = []
    for r in done:
        for row in map(json.loads, (path / f"{r}.jsonl").open()):
            ans = row["response"].get("answers", {})
            rows.append({"review": r, "pmid": row["pmid"], **{f"{prefix}{k}": ans.get(k, {}).get("noul") for k in SECTIONS}})
    return pd.DataFrame(rows, columns=["review", "pmid", *[prefix + k for k in SECTIONS]]).drop_duplicates(["review", "pmid"], keep="last")

for prefix, path in (("p_", OUT), ("f_", FINAL)):
    sub = sub.merge(load_split(path, prefix), on=["review", "pmid"], how="left")
    cols = [prefix + k for k in SECTIONS]
    bad = sub[cols].isna().any(axis=1)
    print(prefix, "records with a failed call or a missing answer (ranked last):", int(bad.sum()))
    sub[f"{prefix}all4"] = sub[cols].prod(axis=1).where(~bad, -1.0)
    sub[f"{prefix}no_outcomes"] = sub[cols[:3]].prod(axis=1).where(~bad, -1.0)

NAMES = {"baseline": "LGAR baseline (reference)", "title_abstract_no_results": "title + abstract w/o results (reference)",
         "f_all4": "protocol criteria only, product all 4", "f_no_outcomes": "protocol criteria only, without outcomes",
         "p_all4": "+ full protocol context, product all 4", "p_no_outcomes": "+ full protocol context, without outcomes"}
results = {NAMES[c]: S.evaluate(sub, c) for c in NAMES}
if done:
    for a in ("all4", "no_outcomes"):
        d = results[NAMES[f"p_{a}"]] - results[NAMES[f"f_{a}"]]
        d["reviews"] = results[NAMES[f"f_{a}"]]["reviews"]
        results[f"context - no context ({a})"] = d
pd.concat(results)

10 of 20 reviews run


baseline: scored 3637 of 3637 records | model: {'typesafe/jev-1.14-20261014'} | cost: $1.80 | without a score (ranked last): 0


title_abstract_no_results: scored 3637 of 3637 records | model: {'typesafe/jev-1.14-20261014'} | cost: $2.06 | without a score (ranked last): 0
p_ records with a failed call or a missing answer (ranked last): 0
f_ records with a failed call or a missing answer (ranked last): 0


MAP  R@1%  R@5%  \
LGAR baseline (reference)                 label_abs      70.7  21.0  60.1   
                                          label_content  77.2  31.4  79.8   
title + abstract w/o results (reference)  label_abs      74.4  22.3  60.4   
                                          label_content  85.4  36.0  84.7   
protocol criteria only, product all 4     label_abs      69.4  19.9  56.6   
                                          label_content  69.8  24.7  71.2   
protocol criteria only, without outcomes  label_abs      68.5  19.6  55.0   
                                          label_content  70.8  26.9  75.6   
+ full protocol context, product all 4    label_abs      69.8  20.7  57.2   
                                          label_content  72.1  28.6  76.5   
+ full protocol context, without outcomes label_abs      68.9  19.7  56.7   
                                          label_content  72.8  26.6  78.6   
context - no context (all4)               label_abs       0.4   0.8   0.6   
                                          label_content   2.3   3.9   5.3   
context - no context (no_outcomes)        label_abs       0.4   0.1   1.7   
                                          label_content   2.0  -0.3   3.0   

                                                         R@10%  R@20%  R@50%  \
LGAR baseline (reference)                 label_abs       73.7   84.8   96.8   
                                          label_content   88.6   94.3   99.2   
title + abstract w/o results (reference)  label_abs       75.8   86.8   99.0   
                                          label_content   91.8   95.9   99.1   
protocol criteria only, product all 4     label_abs       69.4   79.5   96.9   
                                          label_content   85.1   89.0   99.7   
protocol criteria only, without outcomes  label_abs       70.5   79.7   96.3   
                                          label_content   85.4   89.0   99.7   
+ full protocol context, product all 4    label_abs       69.1   81.5   96.8   
                                          label_content   85.2   92.7   99.7   
+ full protocol context, without outcomes label_abs       69.5   78.6   96.3   
                                          label_content   85.8   89.3   99.4   
context - no context (all4)               label_abs       -0.3    2.0   -0.1   
                                          label_content    0.1    3.7    0.0   
context - no context (no_outcomes)        label_abs       -1.0   -1.1    0.0   
                                          label_content    0.4    0.3   -0.3   

                                                         WSS@95%  TNR@95%  \
LGAR baseline (reference)                 label_abs         60.5     71.1   
                                          label_content     79.1     88.0   
title + abstract w/o results (reference)  label_abs         63.8     74.8   
                                          label_content     83.0     92.0   
protocol criteria only, product all 4     label_abs         58.7     69.4   
                                          label_content     80.4     89.6   
protocol criteria only, without outcomes  label_abs         60.5     71.2   
                                          label_content     80.9     90.1   
+ full protocol context, product all 4    label_abs         55.4     65.8   
                                          label_content     80.1     89.1   
+ full protocol context, without outcomes label_abs         53.7     63.9   
                                          label_content     80.7     89.6   
context - no context (all4)               label_abs         -3.3     -3.6   
                                          label_content     -0.3     -0.5   
context - no context (no_outcomes)        label_abs         -6.8     -7.3   
                                          label_content     -0.2     -0.5   

                                                         WSS@100%  reviews  


In [9]:
# per review: protocol vs final, and per-section ablations
per = {}
for r, df in sub.groupby("review"):  # empty until a run has finished
    for label in ("label_abs", "label_content"):
        for c in ("f_no_outcomes", "p_no_outcomes"):
            m = S.metrics(df.sort_values([c, "bm25", "pmid"], ascending=[False, False, True])[label])
            per.setdefault(r, {})[(label, c)] = round(m["MAP"] * 100, 1) if m else None
per = pd.DataFrame(per).T
for label in ("label_abs", "label_content") if len(per) else ():
    d = per[(label, "p_no_outcomes")] - per[(label, "f_no_outcomes")]
    print(f"{label}: with vs without protocol context (without outcomes): better {(d > 0).sum()} / worse {(d < 0).sum()} of {d.notna().sum()} reviews")
if done:
    sec = {}
    for k in SECTIONS:
        for prefix, name in (("f_", "no context"), ("p_", "with context")):
            ev = S.evaluate(sub, prefix + k)
            sec[(k, name)] = {lab: ev.loc[lab, "MAP"] for lab in ("label_abs", "label_content")}
    display(pd.DataFrame(sec).T.rename(columns=lambda c: f"MAP {c} (section alone)"))
per

label_abs: with vs without protocol context (without outcomes): better 5 / worse 5 of 10 reviews
label_content: with vs without protocol context (without outcomes): better 4 / worse 4 of 9 reviews


MAP label_abs (section alone)  \
studies       no context                             38.6   
              with context                           64.8   
participants  no context                             36.0   
              with context                           59.7   
interventions no context                             63.1   
              with context                           67.3   
outcomes      no context                             33.3   
              with context                           61.5   

                            MAP label_content (section alone)  
studies       no context                                 42.0  
              with context                               77.6  
participants  no context                                 26.1  
              with context                               61.9  
interventions no context                                 65.1  
              with context                               69.4  
outcomes      no context                                 23.5  
              with context                               60.9

label_abs               label_content              
         f_no_outcomes p_no_outcomes f_no_outcomes p_no_outcomes
CD000996          85.2          85.6          97.6          94.8
CD001261          54.4          58.6          58.5          77.3
CD004414          73.0          77.8          53.1          58.2
CD007867          47.3          54.0          47.8          54.9
CD010239          90.6          88.8         100.0         100.0
CD011140          85.4          75.0           NaN           NaN
CD011571          80.9          80.2          93.1          87.7
CD011977          81.9          81.2          83.5          81.8
CD012164          22.4          25.5          25.4          25.6
CD012551          64.3          62.6          78.0          75.1

In [10]:
# usage on exactly these records
def usage(path, reviews):
    tok = cost = n = 0
    for r in reviews:
        for row in map(json.loads, (path / f"{r}.jsonl").open()):
            u = row["response"].get("usage", {})
            tok += u.get("input_tokens", 0); cost += u.get("cost") or 0; n += 1
    return {"calls": n, "input_tokens": tok, "usd": round(cost, 3), "tokens_per_call": round(tok / max(n, 1))}

pd.DataFrame({"protocol criteria only": usage(FINAL, done), "+ full protocol context": usage(OUT, done)}).T if done else None

,calls,input_tokens,usd,tokens_per_call
protocol criteria only,3637.0,4683479.0,0.197,1288.0
+ full protocol context,3637.0,23038435.0,0.968,6334.0
